# Fine-tune EnViT5

TV3 phụ trách tích hợp, TV2 hỗ trợ huấn luyện. Bật RUN_TRAIN khi đã cài requirements và có GPU. Notebook dùng tập validation để chọn checkpoint theo loss.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/corpus.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.corpus import records

sample = next(records("train"))
print(sample)

In [ ]:
RUN_TRAIN = False
MODEL = "VietAI/envit5-translation"
MAX_LENGTH = 128

## Huấn luyện có giám sát

Giữ tokenizer gốc và tiền tố en:/vi:. Collator đệm nhãn bằng -100. Sau khi huấn luyện, dùng cùng checkpoint xuất hai tệp dự đoán ở notebook đánh giá.

In [ ]:
if RUN_TRAIN:
    from datasets import Dataset
    from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                              Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)

    set_seed(42)
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

    def encode(batch):
        x = tokenizer(["en: " + s for s in batch["en"]], truncation=True, max_length=MAX_LENGTH)
        x["labels"] = tokenizer(text_target=["vi: " + s for s in batch["vi"]],
                                truncation=True, max_length=MAX_LENGTH)["input_ids"]
        return x

    def prepare(split):
        dataset = Dataset.from_list(list(records(split)))
        return dataset.map(encode, batched=True, remove_columns=dataset.column_names)

    output = root / "artifacts" / "envit5"
    args = Seq2SeqTrainingArguments(
        output_dir=str(output), learning_rate=3e-5, num_train_epochs=3,
        per_device_train_batch_size=4, per_device_eval_batch_size=4,
        gradient_accumulation_steps=8, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, save_total_limit=2, report_to=["tensorboard"], seed=42,
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=prepare("train"),
                             eval_dataset=prepare("validation"),
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
    trainer.train()
    trainer.save_model(str(output / "best"))
    tokenizer.save_pretrained(output / "best")